# Merge modelling panel

Aggregate interim vector / virus tables into supervised spatial–temporal units (specs `03` / `04`).

| Item | Detail |
|------|--------|
| Inputs | `data/02_interim/vectors_normalized.parquet`, `viruses_normalized.parquet` |
| Outputs | `data/03_processed/modelling_panel.parquet` + `modelling_panel_manifest.json` |
| Row semantics (B2) | One row per **observed VectorNet** `(spatial key, year, month, vector_group)` |
| Zone clock (A3) | `first_vector_year` / `first_vector_month` per cell×`vector_group` (from first VectorNet hit) |
| Primary label | `virus_within_5y` (forward ≤5y, same key×`vector_group`) |
| Secondary | `virus_within_10y`; `virus_ever_before` / `virus_ever_after`; nullable delay columns |
| Tier 1 | `(h3_cell, year, month, vector_group)` when VectorNet has H3 |
| Tier 2 | `(country_iso2, year, month, vector_group)` when VectorNet lacks H3 |
| Training filter | **VectorNet coverage required** — virus-only units excluded; no synthetic empty months |
| Ecology | Join only within the same `vector_group` |

Country-only virus rows do **not** invent H3 and do not contribute to Tier‑1 forward / ever / delay labels.

In [7]:
from __future__ import annotations

import hashlib
import json
from pathlib import Path

import pandas as pd
from IPython.display import display

from vector_guard.data import (
    default_processed_dir,
    default_vector_raw_dir,
    default_virus_raw_dir,
    find_repo_root,
)
from vector_guard.resolve import (
    GEO_PRECISION_CENTROID,
    GEO_PRECISION_EXACT,
    H3_RESOLUTION,
    load_interim_vectors,
    load_interim_viruses,
)
from vector_guard.virus_extract import INSDC_SOURCES, OUTBREAK_SOURCES

REPO_ROOT = find_repo_root(Path.cwd())
PROCESSED_DIR = default_processed_dir(REPO_ROOT)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
PANEL_PATH = PROCESSED_DIR / "modelling_panel.parquet"
MANIFEST_PATH = PROCESSED_DIR / "modelling_panel_manifest.json"

REPO_ROOT, PANEL_PATH, MANIFEST_PATH, H3_RESOLUTION

(PosixPath('/workspaces/vector-guard'),
 PosixPath('/workspaces/vector-guard/data/03_processed/modelling_panel.parquet'),
 PosixPath('/workspaces/vector-guard/data/03_processed/modelling_panel_manifest.json'),
 4)

## Load interim sides

In [8]:
vectors = load_interim_vectors()
viruses = load_interim_viruses()

{
    "vectors": len(vectors),
    "viruses": len(viruses),
    "H3_RESOLUTION": H3_RESOLUTION,
}

{'vectors': 120158, 'viruses': 168307, 'H3_RESOLUTION': 4}

## Filter to panel-key-ready rows

Require `vector_group`, `year`, and `month` on both sides. Split Tier 1 (usable `h3_cell`) vs Tier 2 (no H3, usable `country_iso2`). Virus Tier‑1 rows drive B2 forward / ever / delay labels on VectorNet H3 units.

In [9]:
def _ready(df: pd.DataFrame) -> pd.DataFrame:
    return df.loc[
        (df["vector_group"].fillna("") != "")
        & df["year"].notna()
        & df["month"].notna()
    ].copy()


vectors_r = _ready(vectors)
viruses_r = _ready(viruses)

vectors_r["year"] = vectors_r["year"].astype(int)
vectors_r["month"] = vectors_r["month"].astype(int)
viruses_r["year"] = viruses_r["year"].astype(int)
viruses_r["month"] = viruses_r["month"].astype(int)

vectors_t1 = vectors_r.loc[vectors_r["h3_cell"].notna()].copy()
vectors_t2 = vectors_r.loc[
    vectors_r["h3_cell"].isna() & (vectors_r["country_iso2"].fillna("") != "")
].copy()

viruses_t1 = viruses_r.loc[viruses_r["h3_cell"].notna()].copy()
viruses_t2 = viruses_r.loc[
    viruses_r["h3_cell"].isna() & (viruses_r["country_iso2"].fillna("") != "")
].copy()

{
    "vectors_ready": len(vectors_r),
    "vectors_tier1_rows": len(vectors_t1),
    "vectors_tier2_rows": len(vectors_t2),
    "viruses_ready": len(viruses_r),
    "viruses_tier1_rows": len(viruses_t1),
    "viruses_tier2_rows": len(viruses_t2),
}

{'vectors_ready': 119165,
 'vectors_tier1_rows': 119165,
 'vectors_tier2_rows': 0,
 'viruses_ready': 110269,
 'viruses_tier1_rows': 64283,
 'viruses_tier2_rows': 45670}

## Aggregate helpers

Vector side: count features per observed month. Virus side: attach labels **relative to each VectorNet row time `T`** (not same-month-only merge).

- Horizons: `(T, T+5y]` and `(T, T+10y]` (strictly after `T`)
- Ever flags: any virus before / after `T` (0/1)
- Delays: years to first after / since last before (`null` if none on that side)
- QA: `virus_same_month`

In [10]:
HORIZON_YEARS_PRIMARY = 5
HORIZON_YEARS_SECONDARY = 10


def _parse_individual_count(series: pd.Series) -> pd.Series:
    return pd.to_numeric(series, errors="coerce")


def _ym_index(year: pd.Series, month: pd.Series) -> pd.Series:
    return year.astype(int) * 12 + (month.astype(int) - 1)


def _ym_parts(ym: pd.Series) -> tuple[pd.Series, pd.Series]:
    year = ym // 12
    month = (ym % 12) + 1
    return year.astype(int), month.astype(int)


def _mode_country(s: pd.Series) -> str:
    cleaned = s.fillna("").replace("", pd.NA).dropna()
    if cleaned.empty:
        return ""
    return str(cleaned.mode().iloc[0])


def aggregate_vectors_tier1(df: pd.DataFrame) -> pd.DataFrame:
    work = df.copy()
    work["_ic"] = _parse_individual_count(work["individual_count"])
    work["_ym"] = _ym_index(work["year"], work["month"])
    keys = ["h3_cell", "year", "month", "vector_group"]
    out = (
        work.groupby(keys, dropna=False)
        .agg(
            vector_detection_count=("record_id", "size"),
            individual_count_sum=("_ic", "sum"),
            individual_count_mean=("_ic", "mean"),
            country_iso2=("country_iso2", _mode_country),
        )
        .reset_index()
    )
    first = (
        work.groupby(["h3_cell", "vector_group"], dropna=False)["_ym"]
        .min()
        .rename("_first_ym")
        .reset_index()
    )
    fy, fm = _ym_parts(first["_first_ym"])
    first["first_vector_year"] = fy
    first["first_vector_month"] = fm
    out = out.merge(first[["h3_cell", "vector_group", "first_vector_year", "first_vector_month"]], on=["h3_cell", "vector_group"], how="left")
    out["spatial_tier"] = "h3"
    out["vector_present"] = 1
    out["individual_count_sum"] = out["individual_count_sum"].fillna(0.0)
    return out


def aggregate_vectors_tier2(df: pd.DataFrame) -> pd.DataFrame:
    work = df.copy()
    work["_ic"] = _parse_individual_count(work["individual_count"])
    work["_ym"] = _ym_index(work["year"], work["month"])
    keys = ["country_iso2", "year", "month", "vector_group"]
    out = (
        work.groupby(keys, dropna=False)
        .agg(
            vector_detection_count=("record_id", "size"),
            individual_count_sum=("_ic", "sum"),
            individual_count_mean=("_ic", "mean"),
        )
        .reset_index()
    )
    first = (
        work.groupby(["country_iso2", "vector_group"], dropna=False)["_ym"]
        .min()
        .rename("_first_ym")
        .reset_index()
    )
    fy, fm = _ym_parts(first["_first_ym"])
    first["first_vector_year"] = fy
    first["first_vector_month"] = fm
    out = out.merge(
        first[["country_iso2", "vector_group", "first_vector_year", "first_vector_month"]],
        on=["country_iso2", "vector_group"],
        how="left",
    )
    out["spatial_tier"] = "country"
    out["h3_cell"] = pd.NA
    out["vector_present"] = 1
    out["individual_count_sum"] = out["individual_count_sum"].fillna(0.0)
    return out


def _prepare_virus_events(df: pd.DataFrame) -> pd.DataFrame:
    work = df.copy()
    work["_v_ym"] = _ym_index(work["year"], work["month"])
    work["_insdc"] = work["source"].isin(INSDC_SOURCES).astype(int)
    work["_outbreak"] = work["source"].isin(OUTBREAK_SOURCES).astype(int)
    work["_exact"] = (work["geo_precision"] == GEO_PRECISION_EXACT).astype(float)
    work["_centroid"] = (work["geo_precision"] == GEO_PRECISION_CENTROID).astype(float)
    return work


def attach_b2_virus_labels(
    vec_panel: pd.DataFrame,
    viruses: pd.DataFrame,
    *,
    spatial_key: str,
    include_geo_shares: bool,
) -> pd.DataFrame:
    """Attach B2 forward / ever / delay labels to VectorNet units (specs 03/04)."""
    out = vec_panel.copy().reset_index(drop=True)
    out["_row_id"] = out.index
    out["_p_ym"] = _ym_index(out["year"], out["month"])

    empty = {
        "virus_within_5y": 0,
        "virus_within_10y": 0,
        "virus_detection_count_within_5y": 0,
        "virus_detection_count_within_10y": 0,
        "virus_ever_before": 0,
        "virus_ever_after": 0,
        "years_to_first_virus_after": pd.NA,
        "years_since_last_virus_before": pd.NA,
        "virus_same_month": 0,
        "virus_source_n_sequence": 0,
        "virus_source_n_outbreak": 0,
        "share_virus_exact_coords": pd.NA,
        "share_virus_subnational_centroid": pd.NA,
    }
    if viruses.empty:
        for col, val in empty.items():
            out[col] = val
        return out.drop(columns=["_row_id", "_p_ym"])

    vir = _prepare_virus_events(viruses)
    joined = out[["_row_id", spatial_key, "vector_group", "_p_ym"]].merge(
        vir[
            [
                spatial_key,
                "vector_group",
                "_v_ym",
                "record_id",
                "_insdc",
                "_outbreak",
                "_exact",
                "_centroid",
            ]
        ],
        on=[spatial_key, "vector_group"],
        how="left",
    )

    has_virus = joined["record_id"].notna()
    delta_m = joined["_v_ym"] - joined["_p_ym"]
    before = has_virus & (delta_m < 0)
    after = has_virus & (delta_m > 0)
    same = has_virus & (delta_m == 0)
    within_5 = after & (delta_m <= HORIZON_YEARS_PRIMARY * 12)
    within_10 = after & (delta_m <= HORIZON_YEARS_SECONDARY * 12)

    joined["_before"] = before.astype(int)
    joined["_after"] = after.astype(int)
    joined["_same"] = same.astype(int)
    joined["_w5"] = within_5.astype(int)
    joined["_w10"] = within_10.astype(int)
    joined["_delta_after"] = delta_m.where(after)
    joined["_delta_before"] = (-delta_m).where(before)
    joined["_insdc_w5"] = joined["_insdc"].where(within_5, 0)
    joined["_outbreak_w5"] = joined["_outbreak"].where(within_5, 0)
    joined["_exact_w5"] = joined["_exact"].where(within_5)
    joined["_centroid_w5"] = joined["_centroid"].where(within_5)

    agg = joined.groupby("_row_id", sort=False).agg(
        virus_ever_before=("_before", "max"),
        virus_ever_after=("_after", "max"),
        virus_same_month=("_same", "max"),
        virus_within_5y=("_w5", "max"),
        virus_within_10y=("_w10", "max"),
        virus_detection_count_within_5y=("_w5", "sum"),
        virus_detection_count_within_10y=("_w10", "sum"),
        _min_delta_after=("_delta_after", "min"),
        _min_delta_before=("_delta_before", "min"),
        virus_source_n_sequence=("_insdc_w5", "sum"),
        virus_source_n_outbreak=("_outbreak_w5", "sum"),
        share_virus_exact_coords=("_exact_w5", "mean"),
        share_virus_subnational_centroid=("_centroid_w5", "mean"),
    )

    # Rows with no matched virus events: max of empty group still present via left join NaNs
    for col in (
        "virus_ever_before",
        "virus_ever_after",
        "virus_same_month",
        "virus_within_5y",
        "virus_within_10y",
        "virus_detection_count_within_5y",
        "virus_detection_count_within_10y",
        "virus_source_n_sequence",
        "virus_source_n_outbreak",
    ):
        agg[col] = agg[col].fillna(0).astype(int)

    agg["years_to_first_virus_after"] = agg["_min_delta_after"] / 12.0
    agg["years_since_last_virus_before"] = agg["_min_delta_before"] / 12.0
    # When no before/after events, min of all-NaN is NaN — keep as null
    agg = agg.drop(columns=["_min_delta_after", "_min_delta_before"])

    if not include_geo_shares:
        agg["share_virus_exact_coords"] = pd.NA
        agg["share_virus_subnational_centroid"] = pd.NA

    out = out.merge(agg, left_on="_row_id", right_index=True, how="left")
    for col, val in empty.items():
        if col not in out.columns:
            out[col] = val
        elif col in {
            "virus_ever_before",
            "virus_ever_after",
            "virus_same_month",
            "virus_within_5y",
            "virus_within_10y",
            "virus_detection_count_within_5y",
            "virus_detection_count_within_10y",
            "virus_source_n_sequence",
            "virus_source_n_outbreak",
        }:
            out[col] = out[col].fillna(val).astype(int)

    return out.drop(columns=["_row_id", "_p_ym"])

## Build Tier 1 + Tier 2 panels (VectorNet-left join + B2/A3 labels)

In [11]:
vec_t1 = aggregate_vectors_tier1(vectors_t1)
panel_t1 = attach_b2_virus_labels(
    vec_t1,
    viruses_t1,
    spatial_key="h3_cell",
    include_geo_shares=True,
)

vec_t2 = aggregate_vectors_tier2(vectors_t2)
panel_t2 = attach_b2_virus_labels(
    vec_t2,
    viruses_t2,
    spatial_key="country_iso2",
    include_geo_shares=False,
)

panel = pd.concat([panel_t1, panel_t2], ignore_index=True, sort=False)
panel["vector_present"] = 1
panel["country_iso2"] = panel["country_iso2"].fillna("")

column_order = [
    "spatial_tier",
    "h3_cell",
    "country_iso2",
    "year",
    "month",
    "vector_group",
    "vector_present",
    "vector_detection_count",
    "individual_count_sum",
    "individual_count_mean",
    "first_vector_year",
    "first_vector_month",
    "virus_within_5y",
    "virus_within_10y",
    "virus_detection_count_within_5y",
    "virus_detection_count_within_10y",
    "virus_ever_before",
    "virus_ever_after",
    "years_to_first_virus_after",
    "years_since_last_virus_before",
    "virus_same_month",
    "virus_source_n_sequence",
    "virus_source_n_outbreak",
    "share_virus_exact_coords",
    "share_virus_subnational_centroid",
]
panel = panel[column_order].sort_values(
    ["spatial_tier", "vector_group", "year", "month", "h3_cell", "country_iso2"],
    kind="mergesort",
).reset_index(drop=True)

display(panel["spatial_tier"].value_counts())
display(panel.groupby(["spatial_tier", "vector_group"]).size().rename("units").reset_index())
display(
    panel.groupby("spatial_tier")
    .agg(
        units=("vector_present", "size"),
        virus_within_5y=("virus_within_5y", "sum"),
        virus_within_10y=("virus_within_10y", "sum"),
        virus_same_month=("virus_same_month", "sum"),
        virus_ever_after=("virus_ever_after", "sum"),
        virus_ever_before=("virus_ever_before", "sum"),
        virus_positive_rate_5y=("virus_within_5y", "mean"),
    )
)
panel.head()

spatial_tier
h3    12995
Name: count, dtype: int64

,spatial_tier,vector_group,units
0,h3,biting_midge,5038
1,h3,mosquito,6674
2,h3,sandfly,425
3,h3,tick,858


,units,virus_within_5y,virus_within_10y,virus_same_month,virus_ever_after,virus_ever_before,virus_positive_rate_5y
spatial_tier,,,,,,,
h3,12995,1538,2995,45,4754,1289,0.118353


,spatial_tier,h3_cell,country_iso2,year,month,vector_group,vector_present,vector_detection_count,individual_count_sum,individual_count_mean,...,virus_detection_count_within_10y,virus_ever_before,virus_ever_after,years_to_first_virus_after,years_since_last_virus_before,virus_same_month,virus_source_n_sequence,virus_source_n_outbreak,share_virus_exact_coords,share_virus_subnational_centroid
0,h3,8419519ffffffff,GB,1974,3,biting_midge,1,14,11,0.785714,...,0,0,1,42.833333,NaN,0,0,0,NaN,NaN
1,h3,84195b7ffffffff,GB,1974,3,biting_midge,1,14,38,2.714286,...,0,0,1,38.000000,NaN,0,0,0,NaN,NaN
2,h3,8419519ffffffff,GB,1974,4,biting_midge,1,14,15,1.071429,...,0,0,1,42.750000,NaN,0,0,0,NaN,NaN
3,h3,84195b7ffffffff,GB,1974,4,biting_midge,1,21,9,0.428571,...,0,0,1,37.916667,NaN,0,0,0,NaN,NaN
4,h3,8419519ffffffff,GB,1974,5,biting_midge,1,14,1,0.071429,...,0,0,1,42.666667,NaN,0,0,0,NaN,NaN


## Manifest + write

In [12]:
def _sha256(path: Path) -> str | None:
    if not path.exists():
        return None
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


virus_manifest = default_virus_raw_dir(REPO_ROOT) / "manifest.json"
vector_manifest = default_vector_raw_dir(REPO_ROOT) / "manifest.json"

assert panel["vector_present"].eq(1).all()
assert set(panel["spatial_tier"].unique()) <= {"h3", "country"}
assert panel.loc[panel["spatial_tier"] == "h3", "h3_cell"].notna().all()
assert panel.loc[panel["spatial_tier"] == "country", "h3_cell"].isna().all()
assert panel.loc[panel["spatial_tier"] == "country", "country_iso2"].ne("").all()
assert panel["month"].between(1, 12).all()
for col in (
    "virus_within_5y",
    "virus_within_10y",
    "virus_ever_before",
    "virus_ever_after",
    "virus_same_month",
):
    assert panel[col].isin([0, 1]).all()
assert "virus_detected" not in panel.columns
assert (
    panel.loc[panel["virus_ever_after"] == 0, "years_to_first_virus_after"].isna().all()
)
assert (
    panel.loc[panel["virus_ever_before"] == 0, "years_since_last_virus_before"].isna().all()
)
assert (
    panel.loc[panel["virus_ever_after"] == 1, "years_to_first_virus_after"].notna().all()
)
assert (
    panel.loc[panel["virus_ever_before"] == 1, "years_since_last_virus_before"].notna().all()
)
assert (panel["virus_within_5y"] <= panel["virus_within_10y"]).all()
assert (panel["virus_within_10y"] <= panel["virus_ever_after"]).all()

tier_counts = panel["spatial_tier"].value_counts().to_dict()
ym = panel.assign(_ym=panel["year"] * 100 + panel["month"])
ymin_row = ym.loc[ym["_ym"].idxmin()]
ymax_row = ym.loc[ym["_ym"].idxmax()]
date_min = f"{int(ymin_row['year'])}-{int(ymin_row['month']):02d}"
date_max = f"{int(ymax_row['year'])}-{int(ymax_row['month']):02d}"

pos_t1 = (panel["spatial_tier"] == "h3") & (panel["virus_within_5y"] == 1)
virus_geo_mix = {
    "tier1_units_virus_within_5y": int(pos_t1.sum()),
    "tier1_units_virus_within_10y": int(
        ((panel["spatial_tier"] == "h3") & (panel["virus_within_10y"] == 1)).sum()
    ),
    "tier1_units_virus_same_month": int(
        ((panel["spatial_tier"] == "h3") & (panel["virus_same_month"] == 1)).sum()
    ),
    "mean_share_exact_coords_on_positive_tier1": float(
        panel.loc[pos_t1, "share_virus_exact_coords"].dropna().mean()
    )
    if pos_t1.any()
    else None,
    "mean_share_centroid_on_positive_tier1": float(
        panel.loc[pos_t1, "share_virus_subnational_centroid"].dropna().mean()
    )
    if pos_t1.any()
    else None,
}

manifest = {
    "H3_RESOLUTION": H3_RESOLUTION,
    "panel_rows": int(len(panel)),
    "rows_by_spatial_tier": {str(k): int(v) for k, v in tier_counts.items()},
    "eligibility": "from_first_vector",
    "primary_label": "virus_within_5y",
    "secondary_label": "virus_within_10y",
    "delay_missing": "null",
    "horizon_years_primary": HORIZON_YEARS_PRIMARY,
    "horizon_years_secondary": HORIZON_YEARS_SECONDARY,
    "virus_positive_units": int(panel["virus_within_5y"].sum()),
    "virus_positive_rate": float(panel["virus_within_5y"].mean()),
    "virus_within_10y_units": int(panel["virus_within_10y"].sum()),
    "virus_ever_after_units": int(panel["virus_ever_after"].sum()),
    "virus_ever_before_units": int(panel["virus_ever_before"].sum()),
    "virus_same_month_units": int(panel["virus_same_month"].sum()),
    "date_range": {"min": date_min, "max": date_max},
    "vector_groups": sorted(panel["vector_group"].unique().tolist()),
    "training_filter": (
        "Supervised panel includes only observed VectorNet months (B2); "
        "virus-only units are excluded; no synthetic empty months after first vector (A3 clock only). "
        "Primary label virus_within_5y is forward from the row time in the same "
        "spatial key x vector_group. Zeros / null delays mean no reported detection in the merged feeds "
        "(right-censored by extract end), not proven absence."
    ),
    "temporal_grain": "year x month (vector observation); virus labels relative to row timestamp",
    "tier1_key": ["h3_cell", "year", "month", "vector_group"],
    "tier2_key": ["country_iso2", "year", "month", "vector_group"],
    "virus_geo_precision_mix": virus_geo_mix,
    "source_manifest_sha256": {
        "virus_manifest.json": _sha256(virus_manifest),
        "vector_manifest.json": _sha256(vector_manifest),
    },
    "inputs": {
        "vectors_normalized": "data/02_interim/vectors_normalized.parquet",
        "viruses_normalized": "data/02_interim/viruses_normalized.parquet",
    },
    "outputs": {
        "panel": str(PANEL_PATH.relative_to(REPO_ROOT)),
        "manifest": str(MANIFEST_PATH.relative_to(REPO_ROOT)),
    },
}

panel.to_parquet(PANEL_PATH, index=False)
MANIFEST_PATH.write_text(json.dumps(manifest, indent=2, sort_keys=True) + "\n", encoding="utf-8")

print(f"Wrote {PANEL_PATH} ({len(panel):,} rows)")
print(f"Wrote {MANIFEST_PATH}")
display(
    pd.Series(
        {
            k: manifest[k]
            for k in (
                "H3_RESOLUTION",
                "panel_rows",
                "primary_label",
                "virus_positive_units",
                "virus_positive_rate",
                "virus_within_10y_units",
                "virus_ever_after_units",
                "date_range",
            )
        }
    )
)
display(panel.head(12))
display(panel.tail(5))

Wrote /workspaces/vector-guard/data/03_processed/modelling_panel.parquet (12,995 rows)
Wrote /workspaces/vector-guard/data/03_processed/modelling_panel_manifest.json


H3_RESOLUTION                                                4
panel_rows                                               12995
primary_label                                  virus_within_5y
virus_positive_units                                      1538
virus_positive_rate                                   0.118353
virus_within_10y_units                                    2995
virus_ever_after_units                                    4754
date_range                {'min': '1957-09', 'max': '2025-04'}
dtype: object

,spatial_tier,h3_cell,country_iso2,year,month,vector_group,vector_present,vector_detection_count,individual_count_sum,individual_count_mean,...,virus_detection_count_within_10y,virus_ever_before,virus_ever_after,years_to_first_virus_after,years_since_last_virus_before,virus_same_month,virus_source_n_sequence,virus_source_n_outbreak,share_virus_exact_coords,share_virus_subnational_centroid
0,h3,8419519ffffffff,GB,1974,3,biting_midge,1,14,11,0.785714,...,0,0,1,42.833333,NaN,0,0,0,NaN,NaN
1,h3,84195b7ffffffff,GB,1974,3,biting_midge,1,14,38,2.714286,...,0,0,1,38.000000,NaN,0,0,0,NaN,NaN
2,h3,8419519ffffffff,GB,1974,4,biting_midge,1,14,15,1.071429,...,0,0,1,42.750000,NaN,0,0,0,NaN,NaN
3,h3,84195b7ffffffff,GB,1974,4,biting_midge,1,21,9,0.428571,...,0,0,1,37.916667,NaN,0,0,0,NaN,NaN
4,h3,8419519ffffffff,GB,1974,5,biting_midge,1,14,1,0.071429,...,0,0,1,42.666667,NaN,0,0,0,NaN,NaN
5,h3,84195b7ffffffff,GB,1974,5,biting_midge,1,7,4,0.571429,...,0,0,1,37.833333,NaN,0,0,0,NaN,NaN
6,h3,8419519ffffffff,GB,1974,6,biting_midge,1,7,0,0.000000,...,0,0,1,42.583333,NaN,0,0,0,NaN,NaN
7,h3,84195b7ffffffff,GB,1974,6,biting_midge,1,14,4,0.285714,...,0,0,1,37.750000,NaN,0,0,0,NaN,NaN
8,h3,8419519ffffffff,GB,1974,7,biting_midge,1,14,6,0.428571,...,0,0,1,42.500000,NaN,0,0,0,NaN,NaN
9,h3,84195b7ffffffff,GB,1974,7,biting_midge,1,21,4,0.190476,...,0,0,1,37.666667,NaN,0,0,0,NaN,NaN


,spatial_tier,h3_cell,country_iso2,year,month,vector_group,vector_present,vector_detection_count,individual_count_sum,individual_count_mean,...,virus_detection_count_within_10y,virus_ever_before,virus_ever_after,years_to_first_virus_after,years_since_last_virus_before,virus_same_month,virus_source_n_sequence,virus_source_n_outbreak,share_virus_exact_coords,share_virus_subnational_centroid
12990,h3,841fb29ffffffff,FR,2022,6,tick,1,9,27,3.000000,...,0,0,0,NaN,NaN,0,0,0,NaN,NaN
12991,h3,841e037ffffffff,HU,2022,7,tick,1,4,4714,1178.500000,...,0,0,0,NaN,NaN,0,0,0,NaN,NaN
12992,h3,841fb29ffffffff,FR,2022,7,tick,1,9,13,1.444444,...,0,0,0,NaN,NaN,0,0,0,NaN,NaN
12993,h3,841fb29ffffffff,FR,2022,8,tick,1,9,5,0.555556,...,0,0,0,NaN,NaN,0,0,0,NaN,NaN
12994,h3,841fb29ffffffff,FR,2022,12,tick,1,9,34,3.777778,...,0,0,0,NaN,NaN,0,0,0,NaN,NaN
